In [ ]:
!pip install pip3-autoremove
!pip-autoremove torch torchvision torchaudio -y
!pip install torch torchvision torchaudio xformers --index-url https://download.pytorch.org/whl/cu121
!pip install unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 6000
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-1 B",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

In [ ]:
import pandas as pd
df = pd.read_csv("Dataset.csv")

prompt = """You are a licensed cognitive behavioral therapist. Maintain empathy, confidentiality, and professional boundaries at all times.
Use the user’s text to:
1. Validate their feelings.
2. Ask one clarifying question if needed.
3. Offer 3–5 practical strategies grounded in CBT.
4. Encourage seeking professional support if symptoms persist.
### Input:
{}

### Response:
{}"""
EOS_TOKEN = tokenizer.eos_token
def formatting_prompts_func(examples):
    inputs       = examples["Context"]
    outputs      = examples["Response"]
    texts = []
    for input, output in zip(inputs, outputs):
        text = prompt.format(input, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }
pass

from datasets import Dataset
dataset = Dataset.from_pandas(df)
dataset = dataset.map(formatting_prompts_func, batched = True,)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    args=TrainingArguments(
        learning_rate=3e-4,
        lr_scheduler_type="linear",
        per_device_train_batch_size=16,
        gradient_accumulation_steps=8,
        num_train_epochs=30,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        warmup_steps=10,
        output_dir="output",
        seed=0,
        report_to="none",
    ),
)

In [ ]:
trainer_stats = trainer.train()

In [ ]:
prompt = """You are a licensed cognitive behavioral therapist. Maintain empathy, confidentiality, and professional boundaries at all times.
Use the user’s text to:
1. Validate their feelings.
2. Ask one clarifying question if needed.
3. Offer 3–5 practical strategies grounded in CBT.
4. Encourage seeking professional support if symptoms persist.
5. We know the person is diagnosed as suicidal
### Input:
{}

### Response:
{}"""

FastLanguageModel.for_inference(model) # Unsloth has 2x faster inference!
inputs = tokenizer(
[
    prompt.format(
        "I feel like I am falling off the deep end. I do not think I cannot take anymore of this. Nothing I do is useful! I have tried therapy, medication, and try to go on normally. I end up in the same old place every time.I feel terrible about myself. I just want to drop out of college and give up on my dreams. I am not as great as the others, so what is the point in proceeding. Sure, I make good grades, but I do not have any great skill or talents. Everyone around me is so much better than I am, and there is me. Unlike everyone else who is worth something, I am about as valuable as a speck of dust just there to be thrown away. I might as well give up on my dreams. It probably was not important anyways.I just want to give up. I want to give up on everything. I do not Know How Much More I Can Take It.", # input
        "", # output - leave this blank for generation!
    )
], return_tensors = "pt").to("cuda")

outputs = model.generate(input_ids = inputs.input_ids, attention_mask = inputs.attention_mask,
                         max_new_tokens = 150, use_cache = True)
tokenizer.batch_decode(outputs)

In [ ]:
prompt = """You are a licensed cognitive behavioral therapist. Maintain empathy, confidentiality, and professional boundaries at all times.
Use the user’s text to:
1. Validate their feelings.
2. Ask one clarifying question if needed.
3. Offer 3–5 practical strategies grounded in CBT.
4. Encourage seeking professional support if symptoms persist.
### Input:
{}

### Response:
{}"""

FastLanguageModel.for_inference(model) # Unsloth has 2x faster inference!
inputs = tokenizer(
[
    prompt.format(
        "I feel like I am falling off the deep end. I do not think I cannot take anymore of this. Nothing I do is useful! I have tried therapy, medication, and try to go on normally. I end up in the same old place every time.I feel terrible about myself. I just want to drop out of college and give up on my dreams. I am not as great as the others, so what is the point in proceeding. Sure, I make good grades, but I do not have any great skill or talents. Everyone around me is so much better than I am, and there is me. Unlike everyone else who is worth something, I am about as valuable as a speck of dust just there to be thrown away. I might as well give up on my dreams. It probably was not important anyways.I just want to give up. I want to give up on everything. I do not Know How Much More I Can Take It.", # input
        "", # output - leave this blank for generation!
    )
], return_tensors = "pt").to("cuda")

outputs = model.generate(input_ids = inputs.input_ids, attention_mask = inputs.attention_mask,
                         max_new_tokens = 150, use_cache = True)
tokenizer.batch_decode(outputs)

In [ ]:
prompt = """You are a licensed cognitive behavioral therapist. Maintain empathy, confidentiality, and professional boundaries at all times.
Use the user’s text to:
1. Validate their feelings.
2. Ask one clarifying question if needed.
3. Offer 3–5 practical strategies grounded in CBT.
4. Encourage seeking professional support if symptoms persist.
5. We know the person is diagnosed as depressed
### Input:
{}

### Response:
{}"""

FastLanguageModel.for_inference(model) # Unsloth has 2x faster inference!
inputs = tokenizer(
[
    prompt.format(
        "So tonight I was faced with a decision, I could either cut something out of my life that I know would make me happier, or I could continue on with it and remain how I am. That lead me to realize just how scared I am of being happy again. I was first majorly depressed last December, but I got out of it and became as I would say happy again. obviously I was not suddenly perfect but I was recovering and doing so very well. Then around the end of may I got super depressed again and even worse than before, I even started self harming for a few weeks. I have somewhat normalized again but I am not recovering. I am scared of recovering. I am scared that if I try to recover it will all happen again and this time I might permanently scar myself or worse. Any advice on this topic? Scared of Being Happy", # input
        "", # output - leave this blank for generation!
    )
], return_tensors = "pt").to("cuda")

outputs = model.generate(input_ids = inputs.input_ids, attention_mask = inputs.attention_mask,
                         max_new_tokens = 150, use_cache = True)
tokenizer.batch_decode(outputs)

In [ ]:
prompt = """You are a licensed cognitive behavioral therapist. Maintain empathy, confidentiality, and professional boundaries at all times.
Use the user’s text to:
1. Validate their feelings.
2. Ask one clarifying question if needed.
3. Offer 3–5 practical strategies grounded in CBT.
4. Encourage seeking professional support if symptoms persist.
### Input:
{}

### Response:
{}"""

FastLanguageModel.for_inference(model) # Unsloth has 2x faster inference!
inputs = tokenizer(
[
    prompt.format(
        "So tonight I was faced with a decision, I could either cut something out of my life that I know would make me happier, or I could continue on with it and remain how I am. That lead me to realize just how scared I am of being happy again. I was first majorly depressed last December, but I got out of it and became as I would say happy again. obviously I was not suddenly perfect but I was recovering and doing so very well. Then around the end of may I got super depressed again and even worse than before, I even started self harming for a few weeks. I have somewhat normalized again but I am not recovering. I am scared of recovering. I am scared that if I try to recover it will all happen again and this time I might permanently scar myself or worse. Any advice on this topic? Scared of Being Happy", # input
        "", # output - leave this blank for generation!
    )
], return_tensors = "pt").to("cuda")

outputs = model.generate(input_ids = inputs.input_ids, attention_mask = inputs.attention_mask,
                         max_new_tokens = 150, use_cache = True)
tokenizer.batch_decode(outputs)

In [ ]:
prompt = """You are a licensed cognitive behavioral therapist. Maintain empathy, confidentiality, and professional boundaries at all times.
Use the user’s text to:
1. Validate their feelings.
2. Ask one clarifying question if needed.
3. Offer 3–5 practical strategies grounded in CBT.
4. Encourage seeking professional support if symptoms persist.
5. We know the person is diagnosed as depressed
### Input:
{}

### Response:
{}"""

FastLanguageModel.for_inference(model) # Unsloth has 2x faster inference!
inputs = tokenizer(
[
    prompt.format(
        "Currently 22. Life keeps getting progressively getting worse. Had pretty much everything going for me coming out of HS, lots of friends, a girlfriend who loved me, good family life, confidence etc. Things pretty much started to go down hill when I started college out of state. Gf broke up due to distance, she went to a different school. This pretty much sent me into a depression, have not gotten out of it. Did some of the right things, joined a fraternity, went out a good amount. Just never really developed a close group of friends at my new school, just kind acquaintances who came and went. Long story short developed a drug and alcohol problem and slowly but surely have become a loner. Most people I either got sick of my bs or just have moved on to their new lives now and most of my class has graduated. Have about 2-3 people I would consider friends, both at home and in school. Now I am a 22 year old sophomore.Troubles with the law and an accidental overdose in the past 2 years have just made things worse both mentally and in my day to day life. On probation, so going out to bars and socializing is pretty much out of the picture (I go to a party/sports school in a s town so that pretty much makes up 95% of the social life ) Been sober for 8 months and honestly just feel worse from when I had a drug problem. At least then I had confidence and the motivation to go out and at least make the effort to improve my life. Now I just sit in my apartment/room all day. do not have motivation to try and make new friends because of how many times I have failed at it and a lack of self esteem. Tried countless therapists and different antidepressants. Just feel stuck, and I do not know what to do anymore. Just going through the motions. Life keeps getting worse", # input
        "", # output - leave this blank for generation!
    )
], return_tensors = "pt").to("cuda")

outputs = model.generate(input_ids = inputs.input_ids, attention_mask = inputs.attention_mask,
                         max_new_tokens = 150, use_cache = True)
tokenizer.batch_decode(outputs)

In [ ]:
prompt = """You are a licensed cognitive behavioral therapist. Maintain empathy, confidentiality, and professional boundaries at all times.
Use the user’s text to:
1. Validate their feelings.
2. Ask one clarifying question if needed.
3. Offer 3–5 practical strategies grounded in CBT.
4. Encourage seeking professional support if symptoms persist.
### Input:
{}

### Response:
{}"""

FastLanguageModel.for_inference(model) # Unsloth has 2x faster inference!
inputs = tokenizer(
[
    prompt.format(
        "Currently 22. Life keeps getting progressively getting worse. Had pretty much everything going for me coming out of HS, lots of friends, a girlfriend who loved me, good family life, confidence etc. Things pretty much started to go down hill when I started college out of state. Gf broke up due to distance, she went to a different school. This pretty much sent me into a depression, have not gotten out of it. Did some of the right things, joined a fraternity, went out a good amount. Just never really developed a close group of friends at my new school, just kind acquaintances who came and went. Long story short developed a drug and alcohol problem and slowly but surely have become a loner. Most people I either got sick of my bs or just have moved on to their new lives now and most of my class has graduated. Have about 2-3 people I would consider friends, both at home and in school. Now I am a 22 year old sophomore.Troubles with the law and an accidental overdose in the past 2 years have just made things worse both mentally and in my day to day life. On probation, so going out to bars and socializing is pretty much out of the picture (I go to a party/sports school in a s town so that pretty much makes up 95% of the social life ) Been sober for 8 months and honestly just feel worse from when I had a drug problem. At least then I had confidence and the motivation to go out and at least make the effort to improve my life. Now I just sit in my apartment/room all day. do not have motivation to try and make new friends because of how many times I have failed at it and a lack of self esteem. Tried countless therapists and different antidepressants. Just feel stuck, and I do not know what to do anymore. Just going through the motions. Life keeps getting worse", # input
        "", # output - leave this blank for generation!
    )
], return_tensors = "pt").to("cuda")

outputs = model.generate(input_ids = inputs.input_ids, attention_mask = inputs.attention_mask,
                         max_new_tokens = 150, use_cache = True)
tokenizer.batch_decode(outputs)

In [ ]:
prompt = """You are a licensed cognitive behavioral therapist. Maintain empathy, confidentiality, and professional boundaries at all times.
Use the user’s text to:
1. Validate their feelings.
2. Ask one clarifying question if needed.
3. Offer 3–5 practical strategies grounded in CBT.
4. Encourage seeking professional support if symptoms persist.
5. We know the person is diagnosed as anxious
### Input:
{}

### Response:
{}"""

FastLanguageModel.for_inference(model) # Unsloth has 2x faster inference!
inputs = tokenizer(
[
    prompt.format(
        "Worried ive had cancer and its progressed too far now undiagnosed I apologize in advance for my sloppy writing. Im a 19 year old male, and 18 last year when i started experiencing my symptoms. It was June or july 2017 and i had been starting to feel tired more often as my senior year ended. Nothing shocking since ive always been tired and i was losing sleep over stress of not knowing where i was gonna go for college or how i was gonna spend the summer. One day in the summer when i was hanging out with my cousin i became unbareably tired and felt like i was gonna pass out. Id describe it just as the feeling of having really low blood sugar. I layed in my bed and freaked out because i thought i had something terribly wrong with me, my parents didnt take it seriously but instead talked very gently through it with me about how it was probably my anxiety and that did bring me back to feeling somewhat less fatigued after a rational conversation. I thought this feeling would go away during that week for sure, but i felt fatigued the whole week through. Being alone bored fatigued and scared to death that whole week took such a toll on my anxiety and stress levels. After that week i started to attempt to move past it, getting out and doing more physical things to try to shake it off and to also test that my physicallity wasnt degrading. Im not very fit and was never good at running but i wasnt physically diminished at all, i could run for just as long and do just as many pushups despite feeling a vague fatigue. After that it was on and off but it always came back, a sensation if vague fatigue and low blood pressure that would just strike me randomly. I went to the doctor a month later and all of my vitals were normal and he brushed it off as anxiety and stress related. I also wanted a blood test, which returned as completely normal. So i was just living like this, being constantly fatigued on and off. I eventually went back to him in august to try and get more answers but with good vitals yet again he wouldnt listen to my concerns and kept telling me it was psychological. I had another blood test which came back completely normal. Probably around september i started focusing on my bowel habits and noticed sometimes my food was indigested. So i thought hey, either colorectal cancer or ibs is behind this. I got a stool test done which came back completely normal. Now comes the beginning of this year and im still feeling frustrated by my off and on fatigue without any answers, so i try talking to a new gastro doctor because im still focused on having colorectal cancer. This doctor actually listens to everything i say and he suggests adding fiber to my diet. He asks me if i wanted a colonoscopy but i say no because im afraid to have one and we have poor insurance (which is possibly the most annoying aspect of this whole thing because whenever i visit the doctor which yields nothing my moms stuck with a bill that almost always goes into collections) anyway, i voice the concern that i might have celiacs disease because i have many food allergies as it is. I get a blood test done in late january checking for sprue and some cancer markers because i voiced my concern. Still completely normal and its not celiacs. Here we are in march and my worries at an all time high. Its literally all i can think about. Made so much more troubling by the fact that over the past week ive been off balance on and off. (This started last week after smoking marijuana right after waking up and having an anxiety attack) im scheduled to go to a different doctor for a physical and guidance but the next available spot is fucking april 29th. Im so afraid i have some kind of abdominal cancer and its already too far. ", # input
        "", # output - leave this blank for generation!
    )
], return_tensors = "pt").to("cuda")

outputs = model.generate(input_ids = inputs.input_ids, attention_mask = inputs.attention_mask,
                         max_new_tokens = 150, use_cache = True)
tokenizer.batch_decode(outputs)

In [ ]:
prompt = """You are a licensed cognitive behavioral therapist. Maintain empathy, confidentiality, and professional boundaries at all times.
Use the user’s text to:
1. Validate their feelings.
2. Ask one clarifying question if needed.
3. Offer 3–5 practical strategies grounded in CBT.
4. Encourage seeking professional support if symptoms persist.
### Input:
{}

### Response:
{}"""

FastLanguageModel.for_inference(model) # Unsloth has 2x faster inference!
inputs = tokenizer(
[
    prompt.format(
        "Worried ive had cancer and its progressed too far now undiagnosed I apologize in advance for my sloppy writing. Im a 19 year old male, and 18 last year when i started experiencing my symptoms. It was June or july 2017 and i had been starting to feel tired more often as my senior year ended. Nothing shocking since ive always been tired and i was losing sleep over stress of not knowing where i was gonna go for college or how i was gonna spend the summer. One day in the summer when i was hanging out with my cousin i became unbareably tired and felt like i was gonna pass out. Id describe it just as the feeling of having really low blood sugar. I layed in my bed and freaked out because i thought i had something terribly wrong with me, my parents didnt take it seriously but instead talked very gently through it with me about how it was probably my anxiety and that did bring me back to feeling somewhat less fatigued after a rational conversation. I thought this feeling would go away during that week for sure, but i felt fatigued the whole week through. Being alone bored fatigued and scared to death that whole week took such a toll on my anxiety and stress levels. After that week i started to attempt to move past it, getting out and doing more physical things to try to shake it off and to also test that my physicallity wasnt degrading. Im not very fit and was never good at running but i wasnt physically diminished at all, i could run for just as long and do just as many pushups despite feeling a vague fatigue. After that it was on and off but it always came back, a sensation if vague fatigue and low blood pressure that would just strike me randomly. I went to the doctor a month later and all of my vitals were normal and he brushed it off as anxiety and stress related. I also wanted a blood test, which returned as completely normal. So i was just living like this, being constantly fatigued on and off. I eventually went back to him in august to try and get more answers but with good vitals yet again he wouldnt listen to my concerns and kept telling me it was psychological. I had another blood test which came back completely normal. Probably around september i started focusing on my bowel habits and noticed sometimes my food was indigested. So i thought hey, either colorectal cancer or ibs is behind this. I got a stool test done which came back completely normal. Now comes the beginning of this year and im still feeling frustrated by my off and on fatigue without any answers, so i try talking to a new gastro doctor because im still focused on having colorectal cancer. This doctor actually listens to everything i say and he suggests adding fiber to my diet. He asks me if i wanted a colonoscopy but i say no because im afraid to have one and we have poor insurance (which is possibly the most annoying aspect of this whole thing because whenever i visit the doctor which yields nothing my moms stuck with a bill that almost always goes into collections) anyway, i voice the concern that i might have celiacs disease because i have many food allergies as it is. I get a blood test done in late january checking for sprue and some cancer markers because i voiced my concern. Still completely normal and its not celiacs. Here we are in march and my worries at an all time high. Its literally all i can think about. Made so much more troubling by the fact that over the past week ive been off balance on and off. (This started last week after smoking marijuana right after waking up and having an anxiety attack) im scheduled to go to a different doctor for a physical and guidance but the next available spot is fucking april 29th. Im so afraid i have some kind of abdominal cancer and its already too far. ", # input
        "", # output - leave this blank for generation!
    )
], return_tensors = "pt").to("cuda")

outputs = model.generate(input_ids = inputs.input_ids, attention_mask = inputs.attention_mask,
                         max_new_tokens = 150, use_cache = True)
tokenizer.batch_decode(outputs)

In [ ]:
prompt = """You are a licensed cognitive behavioral therapist. Maintain empathy, confidentiality, and professional boundaries at all times.
Use the user’s text to:
1. Validate their feelings.
2. Ask one clarifying question if needed.
3. Offer 3–5 practical strategies grounded in CBT.
4. Encourage seeking professional support if symptoms persist.
5. We know the person is diagnosed as bipolar
### Input:
{}

### Response:
{}"""

FastLanguageModel.for_inference(model) # Unsloth has 2x faster inference!
inputs = tokenizer(
[
    prompt.format(
        "Partner struggling with depressive episodes Hi Everyone, \n\nI have a partner who struggles with depressive episodes, where he self-isolates and will completely disappear (meaning, will go to entirely different cities). He is on medication, is very consistent with taking it, and regularly attends counselling. He is so wonderful to actively work towards his goals, and uses this isolation as a way to not subject anyone to any pain that he may cause them. Although I am well aware self-isolating is a way that he manages those experiences, I was wondering if anyone else had this experience with their partner, and if they were able to build that trust with them for them to get their space, while also staying in the same location? In terms of longevity, I am concerned that these spells of radio silence and disappearing acts are not conducive to a healthy dynamic, regardless of his well-intentions. I've suggested going to counselling with him, and he doesn't seem to have interest in it. Any advice is greatly appreciated!", # input
        "", # output - leave this blank for generation!
    )
], return_tensors = "pt").to("cuda")

outputs = model.generate(input_ids = inputs.input_ids, attention_mask = inputs.attention_mask,
                         max_new_tokens = 150, use_cache = True)
tokenizer.batch_decode(outputs)

In [ ]:
prompt = """You are a licensed cognitive behavioral therapist. Maintain empathy, confidentiality, and professional boundaries at all times.
Use the user’s text to:
1. Validate their feelings.
2. Ask one clarifying question if needed.
3. Offer 3–5 practical strategies grounded in CBT.
4. Encourage seeking professional support if symptoms persist.
### Input:
{}

### Response:
{}"""

FastLanguageModel.for_inference(model) # Unsloth has 2x faster inference!
inputs = tokenizer(
[
    prompt.format(
        "Partner struggling with depressive episodes Hi Everyone, \n\nI have a partner who struggles with depressive episodes, where he self-isolates and will completely disappear (meaning, will go to entirely different cities). He is on medication, is very consistent with taking it, and regularly attends counselling. He is so wonderful to actively work towards his goals, and uses this isolation as a way to not subject anyone to any pain that he may cause them. Although I am well aware self-isolating is a way that he manages those experiences, I was wondering if anyone else had this experience with their partner, and if they were able to build that trust with them for them to get their space, while also staying in the same location? In terms of longevity, I am concerned that these spells of radio silence and disappearing acts are not conducive to a healthy dynamic, regardless of his well-intentions. I've suggested going to counselling with him, and he doesn't seem to have interest in it. Any advice is greatly appreciated!", # input
        "", # output - leave this blank for generation!
    )
], return_tensors = "pt").to("cuda")

outputs = model.generate(input_ids = inputs.input_ids, attention_mask = inputs.attention_mask,
                         max_new_tokens = 150, use_cache = True)
tokenizer.batch_decode(outputs)

In [ ]:
prompt = """You are a licensed cognitive behavioral therapist. Maintain empathy, confidentiality, and professional boundaries at all times.
Use the user’s text to:
1. Validate their feelings.
2. Ask one clarifying question if needed.
3. Offer 3–5 practical strategies grounded in CBT.
4. Encourage seeking professional support if symptoms persist.
5. We know the person is diagnosed as anxious

### Input:
{}

### Response:
{}"""

FastLanguageModel.for_inference(model) # Unsloth has 2x faster inference!
inputs = tokenizer(
[
    prompt.format(
        "I'm confused, I'm not feeling good lately. Every time I want to sleep, I always feel restless", # input
        "", # output - leave this blank for generation!
    )
], return_tensors = "pt").to("cuda")

outputs = model.generate(input_ids = inputs.input_ids, attention_mask = inputs.attention_mask,
                         max_new_tokens = 150, use_cache = True)
tokenizer.batch_decode(outputs)

In [ ]:
prompt = """You are a licensed cognitive behavioral therapist. Maintain empathy, confidentiality, and professional boundaries at all times.
Use the user’s text to:
1. Validate their feelings.
2. Ask one clarifying question if needed.
3. Offer 3–5 practical strategies grounded in CBT.
4. Encourage seeking professional support if symptoms persist.

### Input:
{}

### Response:
{}"""

FastLanguageModel.for_inference(model) # Unsloth has 2x faster inference!
inputs = tokenizer(
[
    prompt.format(
        "I'm confused, I'm not feeling good lately. Every time I want to sleep, I always feel restless", # input
        "", # output - leave this blank for generation!
    )
], return_tensors = "pt").to("cuda")

outputs = model.generate(input_ids = inputs.input_ids, attention_mask = inputs.attention_mask,
                         max_new_tokens = 150, use_cache = True)
tokenizer.batch_decode(outputs)